# 01 — Download and verify

Downloads both dataset versions into Drive and records SHA-256 for every file.

**Credentials:** the auth cell below tries Colab Secrets, then a `kaggle.json` in the
Drive project folder, then a manual upload. Get the token from kaggle.com → Settings →
API → *Create New Token*.

**The critical choice in this notebook** is which mirror of the *original* CICIDS2017 you use.

| Archive | Size | Has Flow ID + IPs? |
|---|---|---|
| `GeneratedLabelledFlows.zip` | 271 MB | yes — this is the one you need |
| `MachineLearningCSV.zip` | 224 MB | no — starts at Destination Port |

Without the IP columns the Arm B match key cannot be built, and Arm B is the paper.
Kaggle mirrors under ~300 MB are usually the ML-CSV variant only.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import config as C
import helpers as H
print('config loaded')

Mounted at /content/drive
config loaded


In [2]:
# Kaggle authentication — tries three routes, in order of preference.
#
#   1. Colab Secrets  (key icon, left sidebar) -> KAGGLE_USERNAME + KAGGLE_KEY
#   2. kaggle.json sitting in the Drive project folder
#   3. one-off manual upload (also saved to Drive so you never repeat it)
#
# Token comes from kaggle.com -> Settings -> API -> Create New Token.

import os, json, shutil

!pip -q install kaggle

CRED_DIR = '/root/.config/kaggle'
os.makedirs(CRED_DIR, exist_ok=True)
CRED = os.path.join(CRED_DIR, 'kaggle.json')
DRIVE_CRED = os.path.join(C.DRIVE_ROOT, 'kaggle.json')


def _write(username, key, persist_to_drive=True):
    payload = {'username': username, 'key': key}
    with open(CRED, 'w') as f:
        json.dump(payload, f)
    os.chmod(CRED, 0o600)
    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key
    if persist_to_drive and not os.path.exists(DRIVE_CRED):
        with open(DRIVE_CRED, 'w') as f:
            json.dump(payload, f)
        print('  (also saved to Drive so future sessions skip this step)')


route = None

# --- route 1: Colab Secrets -------------------------------------------------
try:
    from google.colab import userdata
    u = userdata.get('KAGGLE_USERNAME')
    k = userdata.get('KAGGLE_KEY')
    if u and k:
        _write(u.strip(), k.strip(), persist_to_drive=False)
        route = 'Colab Secrets'
except Exception as e:
    print('Secrets not available:', type(e).__name__)

# --- route 2: kaggle.json already in Drive ----------------------------------
if route is None and os.path.exists(DRIVE_CRED):
    shutil.copy(DRIVE_CRED, CRED)
    os.chmod(CRED, 0o600)
    with open(CRED) as f:
        d = json.load(f)
    os.environ['KAGGLE_USERNAME'] = d['username']
    os.environ['KAGGLE_KEY'] = d['key']
    route = 'kaggle.json in Drive'

# --- route 3: upload it now -------------------------------------------------
if route is None:
    print('No credentials found. Upload the kaggle.json you downloaded:')
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    d = json.loads(up[name].decode())
    _write(d['username'], d['key'])
    route = 'manual upload'

print('\nauthenticated via:', route)

Secrets not available: SecretNotFoundError

authenticated via: kaggle.json in Drive


In [3]:
# Find a suitable mirror of the ORIGINAL dataset.
# Read the output, pick one, then set KAGGLE_ORIGINAL in src/config.py.
!kaggle datasets list -s "CICIDS2017" --max-size 10000000000

ref                                                      title                                             size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  ------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
ericanacletoribeiro/cicids2017-cleaned-and-preprocessed  CICIDS2017: Cleaned & Preprocessed           210143955  2025-01-12 12:02:31.110000           9014         61                1  
mdalamintalukder/cicids2017                              CICIDS2017                                   240838544  2021-05-10 05:15:02.313000           1382          8        0.1764706  
devendra416/ddos-datasets                                DDoS Dataset                                2880543869  2019-04-30 13:49:22.867000          13807        152        0.7058824  
sweety18/cicids2017-full-dataset                         CICIDS2017 Full da

In [4]:
# Scan candidate mirrors for the labelled-flows variant.
# Listing files is cheap; do this before downloading anything.

CANDIDATES = [
    # large enough to plausibly contain BOTH archives (271 MB + 224 MB)
    'pshikk/cicids2017-untampered',
    'raihansultan/cicids2017',
    # mid-size, worth checking
    'bousalihhamza/cicids2017',
    'kk0105/cicids2017',
    'mohanedmohammednaji/cicids2017',
    'shadman1028/cicids2017-official-flow-feature-csv-files',
    'mdalamintalukder/cicids2017',
    'monishapandiyan/cicids2017',
    'naeem41/cicids2017-dataset',
    'sweety18/cicids2017-full-dataset',
]

GOOD_MARKERS = ['trafficlabelling', 'generatedlabelledflows', 'labelled']
BAD_MARKERS  = ['machinelearningcve', 'machinelearningcsv']

try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
except Exception as e:
    api = None
    print('Python API unavailable, will use CLI:', type(e).__name__)


def list_files(slug):
    if api is not None:
        try:
            res = api.dataset_list_files(slug)
            return [f.name if hasattr(f, 'name') else str(f) for f in res.files]
        except Exception:
            pass
    out = os.popen(f'kaggle datasets files -v "{slug}"').read()
    return [ln.split(',')[0] for ln in out.strip().split('\n')[1:] if ln.strip()]


report = []
for slug in CANDIDATES:
    try:
        names = list_files(slug)
    except Exception as e:
        report.append((slug, 'ERROR', str(e)[:60], []))
        continue
    if not names:
        report.append((slug, 'EMPTY', 'no file list returned', []))
        continue
    low = ' '.join(names).lower()
    good = any(m in low for m in GOOD_MARKERS)
    bad = any(m in low for m in BAD_MARKERS)
    verdict = 'USABLE' if good else ('ML-CSV ONLY' if bad else 'UNCLEAR')
    report.append((slug, verdict, f'{len(names)} files', names[:6]))

print(f'{"SLUG":<58} {"VERDICT":<13} DETAIL')
print('-' * 100)
for slug, verdict, detail, sample in report:
    print(f'{slug:<58} {verdict:<13} {detail}')
    for n in sample:
        print(f'{"":<58} {"":<13}   {n}')

usable = [s for s, v, _, _ in report if v == 'USABLE']
print()
print('USABLE mirrors:', usable if usable else 'none found — use the CIC direct download below')

SLUG                                                       VERDICT       DETAIL
----------------------------------------------------------------------------------------------------
pshikk/cicids2017-untampered                               USABLE        16 files
                                                                           GeneratedLabelledFlows/TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
                                                                           GeneratedLabelledFlows/TrafficLabelling /Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
                                                                           GeneratedLabelledFlows/TrafficLabelling /Friday-WorkingHours-Morning.pcap_ISCX.csv
                                                                           GeneratedLabelledFlows/TrafficLabelling /Monday-WorkingHours.pcap_ISCX.csv
                                                                           GeneratedLabelledFlo

### If no Kaggle mirror is usable

The two cells below pull the archive directly from the CIC. This is the canonical source and always works, it is just slower. Skip them if the scan above found a USABLE mirror.

In [5]:
# Fallback: download straight from the Canadian Institute for Cybersecurity.
# This is the canonical source. GeneratedLabelledFlows.zip is ~271 MB (283,876,488 bytes).
#
# Their directory listing is open, so browse it rather than guessing a path:

!curl -sL --max-time 60 "http://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/" | \
    grep -oE 'href="[^"]+"' | head -40

href="//www.unb.ca/webcomps/_css/bootstrap.min.css"
href="//www.unb.ca/webcomps/_css/styles.css"
href="//www.unb.ca/webcomps/_css/color-theme.css"
href="//www.unb.ca/webcomps/_css/webslidemenu.css"
href="//www.unb.ca/webcomps/_font-awesome/css/font-awesome.min.css"
href="//www.unb.ca/webcomps/_css/globalmenustyles-sub-sites.css"
href="#mcontent"
href="#navigation"
href="#audnav"
href="https://phonebook.unb.ca"
href="${cssSitePath.value}"
href="//www.unb.ca/"
href="//www.unb.ca/donations/"
href="https://apply.unb.ca/"
href="https://www.unb.ca"
href="//www.unb.ca/cic/"
href="//www.unb.ca/alumni"
href="//www.unb.ca/cic/"
href="//www.unb.ca/canadian-institute-for-cybersecurity/"
href="//www.unb.ca/cic/about/index.html"
href="//www.unb.ca/cic/about/hub.html"
href="//www.unb.ca/cic/research/index.html"
href="//www.unb.ca/cic/research/applications.html"
href="//www.unb.ca/cic/datasets/android-validation.html"
href="//www.unb.ca/cic/research/pst-conference.html"
href="//www.unb.ca/cic/research

In [6]:
# Once you can see the path from the listing above, fetch the archive.
# Adjust CIC_URL to whatever the listing showed.

CIC_URL = 'http://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/CSVs/GeneratedLabelledFlows.zip'

import os
raw = C.RAW_ORIGINAL          # plain var: shell magic cannot expand $C.ATTR
target = os.path.join(raw, 'GeneratedLabelledFlows.zip')

if not os.path.exists(target):
    !wget --tries=3 --timeout=60 -O "$target" "$CIC_URL"

size = os.path.getsize(target) if os.path.exists(target) else 0
print(f'\ndownloaded {size:,} bytes  (expected about 283,876,488)')

if size > 200_000_000:
    !cd "$raw" && unzip -o -q GeneratedLabelledFlows.zip && rm -f GeneratedLabelledFlows.zip
    print('extracted')
    print(os.listdir(raw))
else:
    print('Download looks wrong or incomplete. Check the URL against the listing above,')
    print('or download it in a browser from unb.ca/cic/datasets/ids-2017.html and')
    print('upload the extracted folder to', raw)


downloaded 108,784 bytes  (expected about 283,876,488)
Download looks wrong or incomplete. Check the URL against the listing above,
or download it in a browser from unb.ca/cic/datasets/ids-2017.html and
upload the extracted folder to /content/drive/MyDrive/research/ids-label-correction/data/raw_original


### Check before you download

In the file listing above you are looking for names like:

```
TrafficLabelling_/Monday-WorkingHours.pcap_ISCX.csv
```

**Good.** These retain Flow ID and IP addresses.

```
MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv
```

**Not usable for Arm B.** These start at `Destination Port` — the IPs are gone.

Once you have a good slug, edit `src/config.py`, set `KAGGLE_ORIGINAL`, and re-run the
config import cell above. If no Kaggle mirror carries the labelled-flows version, download
it directly from the Canadian Institute for Cybersecurity at
`unb.ca/cic/datasets/ids-2017.html` and upload it to Drive by hand — it is a one-off cost.

In [7]:
# Lock in the mirror the scan found, and clear the failed CIC download.
#
# The wget above saved a 108 KB HTML redirect page named GeneratedLabelledFlows.zip.
# It must go: the download cell below skips when the folder is non-empty, so that
# stray file would silently block the real download.

import os, re, importlib

CHOSEN = 'pshikk/cicids2017-untampered'   # USABLE per the scan: GeneratedLabelledFlows/TrafficLabelling_/

# 1. remove anything implausibly small left in raw_original
for fn in os.listdir(C.RAW_ORIGINAL):
    p = os.path.join(C.RAW_ORIGINAL, fn)
    if os.path.isfile(p) and os.path.getsize(p) < 5_000_000:
        print(f'removing stray file: {fn} ({os.path.getsize(p):,} bytes)')
        os.remove(p)

# 2. write the slug into config.py so it survives session restarts
cfg_path = os.path.join(C.DRIVE_ROOT, 'src', 'config.py')
src = open(cfg_path).read()
src = re.sub(r"KAGGLE_ORIGINAL\s*=\s*'[^']*'",
             f"KAGGLE_ORIGINAL = '{CHOSEN}'", src)
open(cfg_path, 'w').write(src)

import config as C
importlib.reload(C)
print('\nKAGGLE_ORIGINAL =', C.KAGGLE_ORIGINAL)
print('raw_original now contains:', os.listdir(C.RAW_ORIGINAL) or '(empty — good)')

removing stray file: GeneratedLabelledFlows.zip (108,784 bytes)

KAGGLE_ORIGINAL = pshikk/cicids2017-untampered
raw_original now contains: (empty — good)


In [8]:
# Download the ORIGINAL
import importlib, config as C
importlib.reload(C)

assert C.KAGGLE_ORIGINAL, 'Set KAGGLE_ORIGINAL in src/config.py first.'

dest = C.RAW_ORIGINAL
if len(os.listdir(dest)) == 0:
    !kaggle datasets download -d "{C.KAGGLE_ORIGINAL}" -p "{dest}" --unzip
else:
    print('already present, skipping download —', len(os.listdir(dest)), 'entries')

Dataset URL: https://www.kaggle.com/datasets/pshikk/cicids2017-untampered
License(s): unknown
100% 507M/507M [00:27<00:00, 19.7MB/s]



In [9]:
# Download the IMPROVED (slug verified: Liu et al., IEEE CNS 2022)
dest = C.RAW_IMPROVED
if len(os.listdir(dest)) == 0:
    !kaggle datasets download -d "{C.KAGGLE_IMPROVED}" -p "{dest}" --unzip
else:
    print('already present, skipping download —', len(os.listdir(dest)), 'entries')

Dataset URL: https://www.kaggle.com/datasets/ernie55ernie/improved-cicids2017-and-csecicids2018
License(s): other
100% 10.2G/10.2G [07:23<00:00, 24.8MB/s]



In [10]:
# After the download: this mirror ships BOTH archives. Confirm what landed,
# and record which subtree the rest of the pipeline must read.

import os

subtrees = {}
for dirpath, dirnames, files in os.walk(C.RAW_ORIGINAL):
    csvs = [f for f in files if f.lower().endswith('.csv')]
    if csvs:
        rel = os.path.relpath(dirpath, C.RAW_ORIGINAL)
        subtrees[rel] = len(csvs)

for k, v in sorted(subtrees.items()):
    tag = ''
    if 'trafficlabelling' in k.lower():
        tag = '   <-- USE THIS (has Flow ID + IPs)'
    elif 'machinelearning' in k.lower():
        tag = '   <-- ignore (no IPs)'
    print(f'{v:>3} csv  {k}{tag}')

labelled = [k for k in subtrees if 'trafficlabelling' in k.lower()]
assert labelled, 'No TrafficLabelling_ folder found — wrong mirror downloaded.'
print('\nOK. Notebooks 02+ will filter to the TrafficLabelling_ subtree.')

  8 csv  GeneratedLabelledFlows/TrafficLabelling    <-- USE THIS (has Flow ID + IPs)
  8 csv  MachineLearningCSV/MachineLearningCVE   <-- ignore (no IPs)

OK. Notebooks 02+ will filter to the TrafficLabelling_ subtree.


In [11]:
# Inventory + SHA-256 manifest. Takes a few minutes on first run; cached after.
import pandas as pd, os, json

manifest_path = os.path.join(C.RESULTS, 'checksums.csv')

rows = []
for version, root in [('original', C.RAW_ORIGINAL), ('improved', C.RAW_IMPROVED)]:
    for dirpath, _, files in os.walk(root):
        for fn in sorted(files):
            if not fn.lower().endswith('.csv'):
                continue
            # original mirror ships both archives; keep only the labelled flows
            if version == 'original' and 'trafficlabelling' not in dirpath.lower():
                continue
            p = os.path.join(dirpath, fn)
            rows.append({
                'version': version,
                'relpath': os.path.relpath(p, root),
                'size_mb': round(os.path.getsize(p) / 1e6, 2),
                'sha256': H.sha256(p),
            })

man = pd.DataFrame(rows)
H.save_table(man, 'checksums.csv')
man

saved /content/drive/MyDrive/research/ids-label-correction/results/checksums.csv (23, 4)


,version,relpath,size_mb,sha256
0,original,GeneratedLabelledFlows/TrafficLabelling /Frida...,96.10,1f779b4f0d78f9225554c4de53b5a2c07912b60dcd136e...
1,original,GeneratedLabelledFlows/TrafficLabelling /Frida...,101.87,7e2ddaa80a5849ba629463296b6128436c161b4a32e803...
2,original,GeneratedLabelledFlows/TrafficLabelling /Frida...,75.39,c061cd98f39d8054aeaed6244a5129a20b3985983d0d02...
3,original,GeneratedLabelledFlows/TrafficLabelling /Monda...,268.65,96f26aea87d513073769b48ce204c2921791149e1c6225...
4,original,GeneratedLabelledFlows/TrafficLabelling /Thurs...,108.72,d74238e054023c8bd4d8056650463cadb25c100fb0e26e...
5,original,GeneratedLabelledFlows/TrafficLabelling /Thurs...,92.03,e3deaff483d18b53b100a441dff9b8919416df8876b4ed...
6,original,GeneratedLabelledFlows/TrafficLabelling /Tuesd...,174.70,ae9c88e10c41a8eb1ff454ae98bc513454925097d0b0b5...
7,original,GeneratedLabelledFlows/TrafficLabelling /Wedne...,285.64,ed538e85b84181e8897dedb3d37d365982f44b27eccd67...
8,improved,CICIDS2017_improved/friday.csv,285.19,ebd499e6f23bd59f9cb81bec28178491b02b925fa5640a...
9,improved,CICIDS2017_improved/monday.csv,207.88,51fe5dc962626efb4ae70dce0303072fb780da0932822b...


In [12]:
# Sanity check: does the original mirror carry the IP columns?
import pandas as pd

orig_csvs = [r for r in rows if r['version'] == 'original']
assert orig_csvs, 'No original CSVs found.'

probe = os.path.join(C.RAW_ORIGINAL, orig_csvs[0]['relpath'])
head = pd.read_csv(probe, nrows=5, encoding='latin-1', low_memory=False)
cols = [H.norm_col(c) for c in head.columns]
print(probe)
print(cols[:12])

has_ip = any('source ip' in c or 'src ip' in c for c in cols)
print()
if has_ip:
    print('PASS — IP columns present. Arm B is possible.')
else:
    print('FAIL — no IP columns. This is the MachineLearningCVE variant.')
    print('       Get GeneratedLabelledFlows / TrafficLabelling_ instead, or Arm B is dead.')

/content/drive/MyDrive/research/ids-label-correction/data/raw_original/GeneratedLabelledFlows/TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
['flow id', 'source ip', 'source port', 'destination ip', 'destination port', 'protocol', 'timestamp', 'flow duration', 'total fwd packets', 'total backward packets', 'total length of fwd packets', 'total length of bwd packets']

PASS — IP columns present. Arm B is possible.


## Gate G1 (part 1)

You have passed this notebook when:

- both `data/raw_original` and `data/raw_improved` contain CSVs,
- `results/checksums.csv` exists,
- the IP-column check above says **PASS**.

Next: `02_audit.ipynb`.